# Sistemas continuos y discretos

:::{important .simple icon=false} **Sistema**
Proceso por el cual las señales de entrada son transformadas, o provocan una respuesta, dando lugar a otras señales como salidas.
:::

Según si las señales de entrada y salida son continuas o discretas, se habla de:

- **Sistemas continuos**: Señales continuas de entrada son transformadas en señales continuas de salida: $x(t)\ \rightarrow y(t)$.

```{figure} figures/T1/2_2_fig1  
---
width: 60%
---
```

- **Sistemas discretos**: Señales discretas de entrada son transformadas en señales discretas de salida: $x[n]\ \rightarrow y[n]$.

```{figure} figures/T1/2_2_fig2 
---
width: 60%
---
```



Además, existen sistemas que tienen entrada continua y salida discreta (muestreadores) y viceversa (interpoladores), que veremos en el tema de muestreo (tema 7).

## Ejemplos sencillos de sistemas

Una de las ventajas del estudio de sistemas es que sistemas muy distintos físicamente producen expresiones matemáticas similares.

**Ejemplos** de sistemas continuos:

- Circuito RC:

```{figure} figures/T1/2_2_fig3 
---
width: 60%
---
```


Considerando $v_s(t)$ como la señal de entrada y $v_c(t)$ como la señal de salida, obtenemos:
```{math}
v_s(t)=R i(t)+v_c(t) \quad \Rightarrow\quad i(t)=\frac{v_s(t)-v_c(t)}{R},
```
```{math}
i(t)=C\frac{d v_c(t)}{dt}.
```
Por tanto, se obtiene la siguiente ecuación diferencial que relaciona la salida con la entrada del sistema:
```{math}
\frac{d v_c(t)}{dt}+\frac{1}{RC}v_c(t)=\frac{1}{RC}v_s(t).
```


In [3]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Select, Div
from bokeh.layouts import column, row

output_notebook(verbose=False, hide_banner=True);

# ==============================================================================
# 1. DATOS PRECALCULADOS
# ==============================================================================

t = np.linspace(0, 10, 2000)
dt = t[1] - t[0]

R_values = np.round(np.linspace(0.2, 5.0, 49), 2)
C_values = np.round(np.linspace(0.2, 5.0, 49), 2)

input_types = ["Escalón", "Seno", "Pulso", "Cuadrada"]

def make_input(t, input_type):
    if input_type == "Escalón":
        return np.ones_like(t)
    elif input_type == "Seno":
        return np.sin(2*np.pi*0.5*t)
    elif input_type == "Pulso":
        return np.where((t >= 1) & (t <= 3), 1.0, 0.0)
    elif input_type == "Cuadrada":
        return np.sign(np.sin(2*np.pi*0.5*t))

def simulate_rc(vs, R, C):
    tau = R*C
    vc = np.zeros_like(vs)

    for n in range(1, len(vs)):
        vc[n] = vc[n-1] + dt*(vs[n-1] - vc[n-1])/tau

    return vc

# Precalculamos solo para R y C iniciales
R_init = 1.0
C_init = 1.0
input_init = "Escalón"

vs_init = make_input(t, input_init)
vc_init = simulate_rc(vs_init, R_init, C_init)

source = ColumnDataSource(data=dict(
    t=t,
    vs=vs_init,
    vc=vc_init
))

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=430,
    width=800,
    tools="pan,wheel_zoom,reset",
    title="Circuito RC: respuesta temporal"
)

p.line("t", "vs", source=source, line_width=3, color="black",
       alpha=0.45, legend_label="Entrada $v_s(t)$")

p.line("t", "vc", source=source, line_width=4, color="blue",
       legend_label="Salida $v_c(t)$")

p.xaxis.axis_label = "t"
p.yaxis.axis_label = "voltaje"
p.grid.grid_line_alpha = 0.25
p.legend.location = "top_right"
p.legend.click_policy = "hide"

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

R_slider = Slider(start=0.2, end=5.0, value=R_init, step=0.1, title="Resistencia R")
C_slider = Slider(start=0.2, end=5.0, value=C_init, step=0.1, title="Capacidad C")

input_select = Select(
    title="Entrada",
    value=input_init,
    options=input_types
)

info = Div(width=800)

info.text = f"""
<div style="font-family:sans-serif; font-size:14px;">
<b>Circuito RC</b><br>
Ecuación del sistema:
<br>
<span style="font-size:18px;">
dv<sub>c</sub>(t)/dt + 1/(RC) v<sub>c</sub>(t) = 1/(RC) v<sub>s</sub>(t)
</span>
<br><br>
Constante de tiempo:
<span style="font-size:18px;">τ = RC = {R_init*C_init:.2f}</span>
</div>
"""

# ==============================================================================
# 4. CALLBACK JAVASCRIPT
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        R_slider=R_slider,
        C_slider=C_slider,
        input_select=input_select,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];
    const vs = data["vs"];
    const vc = data["vc"];

    const R = R_slider.value;
    const C = C_slider.value;
    const tau = R*C;
    const input_type = input_select.value;

    const dt = t[1] - t[0];

    // Entrada v_s(t)
    for (let i = 0; i < t.length; i++) {
        if (input_type === "Escalón") {
            vs[i] = 1.0;
        }
        else if (input_type === "Seno") {
            vs[i] = Math.sin(2*Math.PI*0.5*t[i]);
        }
        else if (input_type === "Pulso") {
            vs[i] = (t[i] >= 1.0 && t[i] <= 3.0) ? 1.0 : 0.0;
        }
        else if (input_type === "Cuadrada") {
            vs[i] = Math.sign(Math.sin(2*Math.PI*0.5*t[i]));
        }
    }

    // Salida v_c(t)
    vc[0] = 0.0;

    for (let i = 1; i < t.length; i++) {
        vc[i] = vc[i-1] + dt*(vs[i-1] - vc[i-1])/tau;
    }

    info.text = `
    <div style="font-family:sans-serif; font-size:14px;">
    <b>Circuito RC</b><br>
    Ecuación del sistema:
    <br>
    <span style="font-size:18px;">
    dv<sub>c</sub>(t)/dt + 1/(RC) v<sub>c</sub>(t) = 1/(RC) v<sub>s</sub>(t)
    </span>
    <br><br>
    Constante de tiempo:
    <span style="font-size:18px;">τ = RC = ${tau.toFixed(2)}</span>
    </div>
    `;

    source.change.emit();
    """
)

R_slider.js_on_change("value", callback)
C_slider.js_on_change("value", callback)
input_select.js_on_change("value", callback)

# ==============================================================================
# 5. LAYOUT
# ==============================================================================

controls = column(input_select, R_slider, C_slider, width=280)

layout = column(
    row(p, controls),
    info
)

show(layout)


- Objeto móvil:

```{figure} figures/T1/2_2_fig4 
```


Si consideramos $f(t)$ como la señal de entrada y $v(t)$ como la señal de salida, $m$ la masa del objeto y $\rho v$ la resistencia por fricción:
```{math}
f(t)=m a(t)+\rho v(t), \qquad a(t)=\frac{d v(t)}{dt}.
```
```{math}
f(t)=m\frac{d v(t)}{dt}+\rho v(t).
```
Se obtiene la siguiente ecuación diferencial para el sistema:
```{math}
\frac{d v(t)}{dt}+\frac{\rho}{m}v(t)=\frac{1}{m}f(t).
```

Ambos ejemplos son matemáticamente equivalentes, pues tienen la misma ecuación diferencial, que podemos escribir:
```{math}
\frac{d y(t)}{dt}+a y(t)=b x(t),
```
siento $x(t)$ la señal de entrada al sistema, $y(t)$ la señal de salida y $a$ y $b$ constantes.


In [6]:
import numpy as np

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, CustomJS, Slider, Select, Div
from bokeh.layouts import column, row

output_notebook(verbose=False, hide_banner=True);

# ==============================================================================
# 1. DATOS INICIALES
# ==============================================================================
t = np.linspace(0, 10, 2000)
dt = t[1] - t[0]

m_init = 1.0
rho_init = 1.0
input_init = "Escalón"

def make_force(t, input_type):
    if input_type == "Escalón":
        return np.ones_like(t)
    elif input_type == "Seno":
        return np.sin(2*np.pi*0.5*t)
    elif input_type == "Pulso":
        return np.where((t >= 1) & (t <= 3), 1.0, 0.0)
    elif input_type == "Cuadrada":
        return np.sign(np.sin(2*np.pi*0.5*t))

def simulate_mass(f, m, rho):
    v = np.zeros_like(f)
    for n in range(1, len(f)):
        v[n] = v[n-1] + dt*(f[n-1] - rho*v[n-1])/m
    return v

f_init = make_force(t, input_init)
v_init = simulate_mass(f_init, m_init, rho_init)

source = ColumnDataSource(data=dict(
    t=t,
    f=f_init,
    v=v_init
))

# ==============================================================================
# 2. FIGURA
# ==============================================================================

p = figure(
    height=430,
    width=800,
    tools="pan,wheel_zoom,reset",
    title="Objeto móvil con fricción: respuesta temporal"
)

p.line("t", "f", source=source, line_width=3, color="black",
       alpha=0.45, legend_label="Entrada f(t)")

p.line("t", "v", source=source, line_width=4, color="blue",
       legend_label="Salida v(t)")

p.xaxis.axis_label = "t"
p.yaxis.axis_label = "amplitud"
p.grid.grid_line_alpha = 0.25
p.legend.location = "top_right"
p.legend.click_policy = "hide"

# ==============================================================================
# 3. CONTROLES
# ==============================================================================

m_slider = Slider(start=0.2, end=5.0, value=m_init, step=0.1, title="Masa m")
rho_slider = Slider(start=0.2, end=5.0, value=rho_init, step=0.1, title="Fricción ρ")

input_select = Select(
    title="Entrada",
    value=input_init,
    options=["Escalón", "Seno", "Pulso", "Cuadrada"]
)

info = Div(width=800)

info.text = f"""
<div style="font-family:sans-serif; font-size:14px;">
<b>Objeto móvil con fricción</b><br>
Ecuación del sistema:
<br>
<span style="font-size:18px;">
dv(t)/dt + ρ/m v(t) = 1/m f(t)
</span>
<br><br>
Constante característica:
<span style="font-size:18px;">τ = m/ρ = {m_init/rho_init:.2f}</span>
</div>
"""

# ==============================================================================
# 4. CALLBACK JAVASCRIPT
# ==============================================================================

callback = CustomJS(
    args=dict(
        source=source,
        m_slider=m_slider,
        rho_slider=rho_slider,
        input_select=input_select,
        info=info
    ),
    code="""
    const data = source.data;
    const t = data["t"];
    const f = data["f"];
    const v = data["v"];

    const m = m_slider.value;
    const rho = rho_slider.value;
    const tau = m/rho;
    const input_type = input_select.value;

    const dt = t[1] - t[0];

    // Entrada f(t)
    for (let i = 0; i < t.length; i++) {
        if (input_type === "Escalón") {
            f[i] = 1.0;
        }
        else if (input_type === "Seno") {
            f[i] = Math.sin(2*Math.PI*0.5*t[i]);
        }
        else if (input_type === "Pulso") {
            f[i] = (t[i] >= 1.0 && t[i] <= 3.0) ? 1.0 : 0.0;
        }
        else if (input_type === "Cuadrada") {
            f[i] = Math.sign(Math.sin(2*Math.PI*0.5*t[i]));
        }
    }

    // Salida v(t)
    // m dv/dt + rho v = f
    // dv/dt = (f - rho v)/m

    v[0] = 0.0;

    for (let i = 1; i < t.length; i++) {
        v[i] = v[i-1] + dt*(f[i-1] - rho*v[i-1])/m;
    }

    info.text = `
    <div style="font-family:sans-serif; font-size:14px;">
    <b>Objeto móvil con fricción</b><br>
    Ecuación del sistema:
    <br>
    <span style="font-size:18px;">
    dv(t)/dt + ρ/m v(t) = 1/m f(t)
    </span>
    <br><br>
    Constante característica:
    <span style="font-size:18px;">τ = m/ρ = ${tau.toFixed(2)}</span>
    <br>
    Para una entrada escalón f(t)=1, la velocidad final tiende a:
    <span style="font-size:18px;">v∞ = 1/ρ = ${(1/rho).toFixed(2)}</span>
    </div>
    `;

    source.change.emit();
    """
)

m_slider.js_on_change("value", callback)
rho_slider.js_on_change("value", callback)
input_select.js_on_change("value", callback)

# ==============================================================================
# 5. LAYOUT
# ==============================================================================

controls = column(input_select, m_slider, rho_slider, width=280)

layout = column(
    row(p, controls),
    info
)

show(layout)


**Ejemplos** de sistemas discretos:

- Cuenta de ahorro en un banco al final de cada año:

Se obtiene la siguiente ecuación en diferencias:
```{math}
y[n]=1.01 y[n-1]+x[n],
```
o de forma equivalente,
```{math}
y[n]-1.01y[n-1]=x[n].
```

- Simulación digital del objeto móvil:

Tomamos el tiempo en intervalos de longitud $\Delta$: 
```{math}
t=n\Delta.
```
Aproximamos la derivada mediante la primera diferencia:
```{math}
\frac{dv(t)}{dt}\simeq\frac{v(n\Delta)-v((n-1)\Delta)}{\Delta}.
```
Por tanto, la ecuación diferencial queda:
```{math}
\frac{v(n\Delta)-v((n-1)\Delta)}{\Delta}+\frac{\rho}{m}v(n\Delta)=\frac{1}{m}f(n\Delta),
```
usando las siguientes definiciones de señales discretas a partir de las continuas muestreadas:
```{math}
\begin{cases}v[n]=v(n\Delta),\\f[n]=f(n\Delta),\end{cases}
```
obtenemos:
```{math}
v[n]-v[n-1]+\frac{\rho\Delta}{m}v[n]=\frac{\Delta}{m}f[n],
```
y agrupando términos:
```{math}
v[n]\frac{m+\rho\Delta}{m}-v[n-1]=\frac{\Delta}{m}f[n].
```
Si finalmente dividimos la ecuación por el factor $(m+\rho\Delta)/m$:
```{math}
v[n]-\frac{m}{m+\rho\Delta}v[n-1]=\frac{\Delta}{m+\rho\Delta}f[n].
```

Se puede observar que ambos son ejemplos del mismo tipo de sistema, ya que tienen la misma ecuación en diferencias:
```{math}
y[n]-a y[n-1]=b x[n].
```




## Sistemas elementales (transformación de la variable independiente)
Un concepto fundamental en el análisis de señales y sistemas es el de la transformación de una señal mediante un cierto sistema. Vamos a ver algunas transformaciones elementales de señales, que serán muy importantes en el resto de la asignatura, y nos permitirán entender mejor algunas propiedades de las señales, ya vistas, como son las señales pares e impares, reales e imaginarias, hermíticas y antihermíticas (ver seccción 2. [](#clases_senales)), señales periódicas (ver sección 2. [](#periodicas)), y de sistemas, que veremos en la sección 2. [](#propiedades_sistemas).

En primer lugar veremos algunas transformaciones elementales sobre la amplitud de la señal, o sumas y diferencias sobre la señal:

- **Cambio de nivel**: multiplicación de la señal por una constante.

```{figure} figures/T1/2_2_fig5_a 
---
width: 60%
---
```

```{figure} figures/T1/2_2_fig5_b 
---
width: 60%
---
```


Amplificación: $A>1$.
Atenuación: $A<1$.

- **Conjugación**:

```{figure} figures/T1/2_2_fig6_a 
---
width: 60%
---
```

```{figure} figures/T1/2_2_fig6_b 
---
width: 60%
---
```


Se ha usado en señales reales e imaginarias y hermíticas y antihermíticas.

- **Integración**:

```{figure} figures/T1/2_2_fig7_a 
---
width: 60%
---
```



- **Sumación o acumulación**: equivalente a la integración para señales discretas.

```{figure} figures/T1/2_2_fig7_b 
---
width: 60%
---
```



- **Derivación**: operación inversa a la integración.

```{figure} figures/T1/2_2_fig8_a 
---
width: 60%
---
```



- **Diferenciación o primera diferencia**: operación inversa a la sumación.

```{figure} figures/T1/2_2_fig8_b 
---
width: 70%
---
```


A continuación veremos operaciones elementales realizadas sobre la variable independiente.

- **Desplazamiento o corrimiento en el tiempo**:
  
  - Continuo:
  
```{figure} figures/T1/2_2_fig9_a 
---
width: 70%
---
```


  - Discreto:
  
```{figure} figures/T1/2_2_fig9_b 
```

  
  
Se ha usado para definir las señales periódicas.

- **Inversión en el tiempo o abatimiento**: reflexión respecto al origen de tiempos.

```{figure} figures/T1/2_2_fig10 
```


Se ha usado para definir señales pares e impares y hermíticas y antihermíticas.
**Ejemplo**: cinta escuchada al revés.
  
- **Cambio de escala**:
  
  - Continuo:
  
```{figure} figures/T1/2_2_fig11_a 
```

  
Si $x(t_0)=0,\quad at=t_0\ \Rightarrow\ t=t_0/a$.
  
  - Discreto:
  
```{figure} figures/T1/2_2_fig11_b 
---
width: 75%
---
```


  La expansión discreta o inserción de ceros se define de la siguiente forma:
  \begin{equation}
    x[n/k]\overset{\Delta}{=}\begin{cases}
x[n/k], & n=Mk,\\
0, & \text{resto.}
    \end{cases},\qquad M\in\Z.
  \end{equation}
**Ejemplo**: cambio de velocidad en disco de vinilo.
- **Transformación lineal del eje de tiempos**:

```{figure} figures/T1/2_2_fig12 
---
width: 60%
---
```


Conserva la forma de la señal, pero:

  - $|\alpha|<1$: alarga linealmente la señal.
  - $|\alpha|<1$: comprime linealmente la señal.
  - $\alpha<0$: invierte en el tiempo la señal.
  - $\beta\neq 0$: desplaza en el tiempo la señal.

Forma de hacerlo gráficamente de forma sistemática[^1]:
 [^1]:Se puede hacer en el orden inverso pero hay que tener cuidado.

  - Desplazamiento: $x(t)\ \rightarrow\ x(t+\beta)$.
  - Escalamiento y/o inversión: $x(t+\beta)\ \rightarrow\ x(\alpha t+\beta)$.


**Ejemplo**: $x(t)\ \rightarrow\ x\left(\frac{3}{2}t+1\right).$

```{figure} figures/T1/2_2_fig13 
---
width: 60%
---
```


Podemos comprobar que en ciertos instantes temporales de interés el resultado es correcto:

-  $t=-\frac{2}{3}\ \Rightarrow\ x\left(\frac{3}{2}\cdot\left(-\frac{2}{3}\right)+1\right)=x(-1+1)=x(0);$
-  $t=0\ \Rightarrow\ x\left(\frac{3}{2}\cdot 0+1\right)=x(1);$
-  $ t=\frac{2}{3}\ \Rightarrow\ x\left(\frac{3}{2}\cdot\frac{2}{3}+1\right)=x(1+1)=x(2).$

